# 1. Project Overview
Rihla supports learning modern history (1700 onward). This notebook demonstrates its finalized category assessment component using administrator-assigned question difficulty and corrected trivia responses.

This is a configured IRT-based statistical estimator: no shared model parameters are trained. The complete research history remains in `archive/Rihla_ML_Weakness_Detection_Research.ipynb`; the original notebook is retained. All demonstration records below are **synthetic**, not evidence of real historical assessment performance.

# 2. Final Assessment Architecture
Administrator assigns Easy/Medium/Hard → user completes trivia → fixed-difficulty IRT estimates ability per category → numerical evidence supports future strength/weakness reporting.

Question labels and their historical versions are supplied by the application; inference never learns or edits them. Actual accuracy, smoothed accuracy and latent ability are separate measurements. No approved qualitative cutoffs exist, so the component returns numerical estimates and explicit evidence/availability statuses without assigning Strong/Weak labels.

# 3. Imports and Configuration
Only Python's standard library is required. Run from the repository or place the two inference/test files together in Colab's `/content`. A mounted Drive copy under `Rihla_ML/ml/knowledge_assessment` is also supported. Mount Drive separately if needed; this notebook does not download data, install packages or run research models.

In [1]:
import copy
import importlib.util
import io
import json
from pathlib import Path
import sys
import tempfile
import unittest

candidates = [Path.cwd(), Path('/content'),
              Path('/content/drive/MyDrive/Rihla_ML/ml/knowledge_assessment')]
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.append(parent / 'ml/knowledge_assessment')
module_dir = next((p for p in candidates if (p / 'fixed_difficulty_irt.py').is_file()), None)
if module_dir is None:
    raise FileNotFoundError('Provide fixed_difficulty_irt.py and test_fixed_difficulty_irt.py from ml/knowledge_assessment; no research artifacts are required.')
sys.path.insert(0, str(module_dir.resolve()))
import fixed_difficulty_irt as irt
import test_fixed_difficulty_irt as assessment_tests

CONFIG = {'difficulty_mapping': dict(irt.DEFAULT_MAPPING),
          'prior_config': dict(irt.DEFAULT_PRIOR),
          **irt.DEFAULT_ASSESSMENT,
          'data_context': 'synthetic_functional_demo'}
print('Configured estimator:', irt.MODEL_VERSION, '| third-party dependencies: none')

Configured estimator: fixed-1pl-map-v1 | third-party dependencies: none


# 4. Input Data Structure
Each answer requires `user_id`, unique `attempt_id`, `game_id`, `question_id`, `category_id`, `category_name`, exact `difficulty` label, binary `answered_correctly`, timezone-aware `answered_at`, and `game_status`. Supply `question_version` and `feedback_available_at` to preserve administrator snapshots and actual feedback timing.

Game records contain `user_id`, unique `game_id`, valid `status`, and `completed_at` for completed games; optional `question_count` must match the supplied history. The high-level assessment validates complete game histories and defaults missing feedback time to game completion. The low-level updater assumes the caller has already validated completion and released feedback. IDs are returned as strings. Exact response retries are ignored; conflicting retries are rejected. Same-question repeats within one game are invalid.

In [2]:
example_games, example_answers = assessment_tests.synthetic_fixture()
validated_answer = irt.normalize_answer(example_answers[0])
print(json.dumps(validated_answer, indent=2))

{
  "user_id": "1",
  "attempt_id": "101",
  "game_id": "1",
  "question_id": "1",
  "category_id": "2",
  "category_name": "Wars",
  "difficulty": "Easy",
  "answered_correctly": 0,
  "answered_at": "2026-01-01T11:41:00.000000+00:00",
  "feedback_available_at": "2026-01-01T12:00:00.000000+00:00",
  "game_status": "COMPLETED",
  "question_version": "admin-v1"
}


# 5. Manual Difficulty Mapping
The initial mapping is **Easy = −1, Medium = 0, Hard = +1**. These configurable values are provisional assumptions, not empirically calibrated question parameters. Invalid or missing labels are rejected. A configuration fingerprint prevents silent mapping/prior changes in stored state; changing them requires reconstruction. Administrator edits require a new question version, retaining past response snapshots.

In [3]:
mapping, prior = irt.normalize_configuration(CONFIG['difficulty_mapping'], CONFIG['prior_config'])
assert mapping == {'Easy': -1.0, 'Medium': 0.0, 'Hard': 1.0}
print(mapping)

{'Easy': -1.0, 'Medium': 0.0, 'Hard': 1.0}


# 6. Fixed-Difficulty IRT
For category ability $\theta$ and fixed administrator difficulty $b$,
$$P(Y=1\mid\theta,b)=\operatorname{sigmoid}(\theta-b).$$
The reused module estimates the maximum a posteriori (MAP) ability with a fixed $N(0,1)$ prior by default. It solves the one-dimensional strictly convex negative log posterior using safeguarded Newton iterations:
$$\frac{\theta-\mu}{\sigma^2}+\sum_{d\in\{E,M,H\}}[n_d\operatorname{sigmoid}(\theta-b_d)-c_d]=0.$$
Only three difficulty-specific response/correct counts are needed for each category's likelihood. The Gaussian prior keeps all-correct, all-incorrect and sparse histories finite. No question parameters are optimized. Conditional SE is inverse square root posterior curvature; it ignores mapping uncertainty and response dependence.

Reconstruction sorts by feedback time, answer time and attempt ID. The incremental updater rejects new late events and accepts identical old retries. Same-time feedback predictions use a frozen pre-block state. Because this estimator assumes constant ability within the supplied history, the final count-based estimate is order-invariant; chronology still governs intermediate availability. It is not a learning/forgetting model.

In [4]:
first = example_answers[0]
initial_state = irt.new_user_state(first['user_id'], mapping, prior)
updated_state = irt.update_user_ability(initial_state, first)
rebuilt_state = irt.estimate_user_ability([first], mapping, prior)
assert updated_state == rebuilt_state
assert initial_state['applied_count'] == 0
assert updated_state['difficulty_mapping'] == mapping
print('One-response private ability (logits):', updated_state['categories'][str(first['category_id'])]['theta'])

One-response private ability (logits): -0.5989418624585936


# 7. Category-Level Knowledge Assessment
`analyze_historical_knowledge(user_id, game_records, interactions, configuration)` is the public entry point. It validates games, selects the requested user, reconstructs independent category states and applies availability rules. Low-level ability functions return private internal state and must not bypass the public eligibility gate.

Categories use real application identifiers, with no Riiid mapping. No global ability is substituted for category ability. Numerical comparisons can inform future strengths/weakness reporting after policy validation; no automated qualitative classification is currently enabled.

In [5]:
analyze_historical_knowledge = irt.analyze_historical_knowledge
assessment = analyze_historical_knowledge(1, example_games, example_answers, CONFIG)
assert assessment == irt.analyze_fixed_historical_knowledge(1, example_games, example_answers, CONFIG)
print('Category count:', len(assessment['knowledge_assessment']))

Category count: 2


# 8. Performance Statistics
Actual accuracy is $100c/n$ percent. Smoothed accuracy is $(c+lpha)/(n+lpha+eta)$, a descriptive fraction with inherited $lpha=eta=5$. Ability is in **logits**, never a knowledge percentage.

Preserve the existing provisional category evidence criteria: at least ten responses across three valid completed games. They are configurable reporting assumptions, not approved weakness cutoffs. Sparse categories return `INSUFFICIENT_EVIDENCE` and null public ability. Evidence includes conditional SE, counts by difficulty and repeated-question attempts. Repeated questions across games count separately but may overstate independent evidence.

In [6]:
for row in assessment['knowledge_assessment']:
    n, c = row['total_attempts'], row['correct_answers']
    assert row['accuracy_percent'] == 100 * c / n
    assert row['smoothed_accuracy'] == (c + 5) / (n + 10)
    print(row['category_name'], {'accuracy_percent': row['accuracy_percent'],
          'smoothed_fraction': round(row['smoothed_accuracy'], 6),
          'ability_logits': round(row['estimated_irt_ability'], 6),
          'status': row['assessment_status']})

Wars {'accuracy_percent': 52.0, 'smoothed_fraction': 0.514286, 'ability_logits': -0.101296, 'status': 'AVAILABLE_PROVISIONAL_MAPPING'}
Politics {'accuracy_percent': 68.0, 'smoothed_fraction': 0.628571, 'ability_logits': 0.742493, 'status': 'AVAILABLE_PROVISIONAL_MAPPING'}


# 9. Five-Game Eligibility
Assessment becomes visible only after **five valid completed games**. A valid completed game contains exactly ten distinct answered questions. Abandoned, incomplete and short games do not count; duplicate game records and malformed records are rejected. Identical response retries are deduplicated before counting completeness. Overall eligibility does not establish evidence sufficiency for every category.

In [7]:
four_game_answers = [a for a in example_answers if a['game_id'] != 5]
four_game_result = analyze_historical_knowledge(1, example_games[:4], four_game_answers, CONFIG)
assert four_game_result['eligible'] is False
assert four_game_result['knowledge_assessment'] == []
assert assessment['eligible'] and assessment['completed_games'] == 5
print('Four completed games: hidden; five valid completed games: eligible.')

Four completed games: hidden; five valid completed games: eligible.


# 10. Example Rihla Assessment
The following small **synthetic functional demonstration** contains two users, three historical categories, all three manual difficulty labels, both outcomes and an abandoned game. User 1 has five valid games but sparse Industrialization evidence. User 2 has four valid completed games and one abandoned game, so no category estimates are exposed. These records do not validate real-world predictive performance.

In [8]:
synthetic_games, synthetic_answers = assessment_tests.synthetic_fixture()
synthetic_games, synthetic_answers = copy.deepcopy(synthetic_games), copy.deepcopy(synthetic_answers)
# Two distinct synthetic questions create a deliberately sparse category.
for answer in synthetic_answers:
    if answer['question_id'] == 1 and answer['game_id'] <= 2:
        answer.update(category_id=4, category_name='Industrialization',
                      question_id=1000 + answer['game_id'])
other_games, other_answers = assessment_tests.synthetic_fixture()
for game in other_games:
    game['user_id'] = 2
    if game['game_id'] == 5:
        game.update(status='ABANDONED', completed_at=None)
for answer in other_answers:
    answer['user_id'] = 2
    if answer['game_id'] == 5:
        answer['game_status'] = 'ABANDONED'
synthetic_games.extend(other_games)
synthetic_answers.extend(other_answers)
synthetic_results = {str(user): analyze_historical_knowledge(
    user, synthetic_games, synthetic_answers, CONFIG) for user in (1, 2)}
assert synthetic_results['1']['eligible']
assert not synthetic_results['2']['eligible']
sparse = next(r for r in synthetic_results['1']['knowledge_assessment'] if r['category_id'] == '4')
assert sparse['assessment_status'] == 'INSUFFICIENT_EVIDENCE' and sparse['estimated_irt_ability'] is None
print('SYNTHETIC ONLY: 2 users, 100 responses, 9 completed games and 1 abandoned game')
print(json.dumps(synthetic_results, indent=2, allow_nan=False))

SYNTHETIC ONLY: 2 users, 100 responses, 9 completed games and 1 abandoned game
{
  "1": {
    "user_id": "1",
    "eligible": true,
    "completed_games": 5,
    "games_remaining": 0,
    "model_version": "fixed-1pl-map-v1",
    "configuration_sha256": "a02964fbfeeb52d47fa8385dc324941e7385e6232fe7c1714d4dec7ad7f180ff",
    "model_type": "configured_fixed_1pl_map",
    "shared_parameters_fitted": false,
    "mapping_empirically_calibrated": false,
    "data_context": "synthetic_functional_demo",
    "knowledge_assessment": [
      {
        "category_id": "2",
        "category_name": "Wars",
        "total_attempts": 23,
        "correct_answers": 13,
        "accuracy_percent": 56.52173913043478,
        "smoothed_accuracy": 0.5454545454545454,
        "smoothed_accuracy_units": "fraction; descriptive statistic, not mastery",
        "estimated_irt_ability": 0.13374239212404615,
        "ability_units": "logits on the configured provisional scale",
        "completed_games": 5,
      

# 11. Testing and Validation
Reuse the existing focused 24-test suite rather than duplicating it. It covers fixed mapping, update directions, finite extreme/sparse histories, chronology, current/future-label independence, same-feedback blocks, retry idempotency/conflicts, reconstruction, category independence, versioned question snapshots, five/short/abandoned/incomplete games, malformed inputs and JSON reload consistency. Section 12 additionally checks independently imported export parity. All tests use small synthetic records; failures stop execution.

In [9]:
test_stream = io.StringIO()
suite = unittest.defaultTestLoader.loadTestsFromModule(assessment_tests)
test_result = unittest.TextTestRunner(stream=test_stream, verbosity=1).run(suite)
print(test_stream.getvalue())
assert test_result.wasSuccessful() and not test_result.skipped
print(f'{test_result.testsRun} tests passed; no failures, errors or skips.')

........................
----------------------------------------------------------------------
Ran 24 tests in 0.098s

OK

24 tests passed; no failures, errors or skips.


# 12. Export and Integration Preparation
The reusable module is `ml/knowledge_assessment/fixed_difficulty_irt.py`; the notebook imports that source instead of maintaining another implementation. The functions `estimate_user_ability`, `update_user_ability` and `analyze_historical_knowledge` return JSON-serializable values. `serialize_state`/`load_state` preserve configuration and retry state. The cell below copies the module into a temporary directory, imports it independently and checks assessment/state parity; it creates no persistent model artifact and overwrites no research export.

Future Spring Boot integration should supply authorized corrected responses and complete game histories with administrator label/category/version snapshots and feedback timestamps. Persist returned private state and consumed attempt IDs atomically. A late event or changed mapping/prior requires history reconstruction. This task implements no endpoint, database, UI or service integration.

In [10]:
with tempfile.TemporaryDirectory(prefix='rihla_assessment_check_') as temporary:
    exported_path = Path(temporary) / 'fixed_difficulty_irt.py'
    exported_path.write_bytes((module_dir / 'fixed_difficulty_irt.py').read_bytes())
    spec = importlib.util.spec_from_file_location('exported_rihla_assessment', exported_path)
    exported = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(exported)
    reloaded = exported.analyze_historical_knowledge(1, synthetic_games, synthetic_answers, CONFIG)
    assert reloaded == synthetic_results['1']
    restored = exported.load_state(irt.serialize_state(updated_state))
    assert restored == updated_state
    assert exported.update_user_ability(restored, first) == restored
    encoded = json.dumps(reloaded, allow_nan=False)
    assert json.loads(encoded) == reloaded
print('Export/import, assessment, JSON state and duplicate retry parity passed.')

Export/import, assessment, JSON state and duplicate retry parity passed.


# 13. Findings and Limitations
The component reuses the verified configured fixed-difficulty MAP estimator, independent category states, descriptive statistics, validated five-game gating and explicit sparse-evidence handling. Question difficulty remains administrator-owned. No new research model or shared parameters were trained, and no large dataset is required.

**The original automatic-difficulty research model's 71.50% prediction accuracy does not apply to this fixed-difficulty implementation.** Synthetic functional results establish software behavior only. Real historical predictive/knowledge validity remains unverified. No qualitative strength/weakness thresholds are approved; the current output deliberately reports numerical evidence and availability.

The −1/0/+1 mapping, Gaussian ability prior, smoothing prior and inherited category evidence criteria require real-data validation. Conditional SE omits mapping error, memorization/dependence and changes in ability over time. Cross-category absolute comparisons assume comparable administrator label semantics. The private deduplication ledger grows with history; storage/concurrency/authentication are application responsibilities.

**Next step:** collect real Rihla historical response histories with independently reviewed administrator labels and validated completion/feedback metadata, then validate assessment interpretation and approve reporting policy before integration.